In [22]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.max_colwidth", None)  # show full width of showing cols
pd.set_option(
    "display.expand_frame_repr", False
)  # print cols side by side as it's supposed to be


ODIS_DUCKDB_FILE = "odis.duckdb"
PCC_DUCKDB_FILE = "dev.duckdb"

con = duckdb.connect(database=PCC_DUCKDB_FILE, read_only=True)
con.sql(f"ATTACH '{ODIS_DUCKDB_FILE}' AS odis;")

BinderException: Binder Error: Failed to attach database: database with name "odis" already exists

In [23]:
# informacion de comunas y riesgos
query_catnat = """
SELECT *
  FROM dev.main.catnat_gaspar
  ORDER BY cod_commune ASC;
"""

cat_nat = con.sql(query_catnat)
cat_nat_df = cat_nat.df()

query_odis3 = """SELECT * FROM odis.gold_gold_typologies_territoires
ORDER BY codgeo ASC;"""
odis_topology = con.sql(query_odis3).df()

chomage_df = pd.read_csv(
    r"C:\Users\marin\OneDrive\Documentos\GitHub\14_PrixChangementClimatique\data\utils\downloaded_files\taux_chomage_communes.csv"
)


In [28]:
from datetime import datetime

climate_data = pd.DataFrame(
    {
        "lib_commune": cat_nat_df["lib_commune"],
        "cod_commune": cat_nat_df["cod_commune"],
        "climate_risk": cat_nat_df["lib_risque_jo"],
        "durée événement": pd.to_datetime(cat_nat_df["dat_fin"], format="mixed")
        - pd.to_datetime(cat_nat_df["dat_deb"], format="mixed"),
        "cat_nat_recognition": cat_nat_df["dat_pub_jo"],
        "recognition_time": pd.to_datetime(cat_nat_df["dat_pub_jo"], format="mixed")
        - pd.to_datetime(cat_nat_df["dat_fin"], format="mixed"),
    }
)
climate_df = (
    climate_data.groupby(["cod_commune", "lib_commune", "climate_risk"])
    .agg(
        duree_event_mean=("durée événement", "mean"),
        duree_event_max=("durée événement", "max"),
        nb_events=("durée événement", "count"),
        time_recognition_mean=("recognition_time", "mean"),
        time_recognition_max=("recognition_time", "max"),
    )
    .reset_index()
)

In [ ]:
merged_df = pd.merge(
    climate_df, chomage_df, left_on="cod_commune", right_on="codgeo", how="inner"
)